In [ ]:
# --- setup -------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

import os

REPO = '/content/pxr-repo'
if os.path.exists(REPO):
    !cd $REPO && git pull
else:
    !git clone https://github.com/pridem755/patient-or-xray.git $REPO
%pip install -q -e $REPO

In [ ]:
import importlib
import site
import sys

site.main()
importlib.invalidate_caches()
SRC = f'{REPO}/src'
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import pxr
print('pxr loaded from:', pxr.__file__)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from pxr.config import load_config
from pxr.stats.fairness import (
    analyse_label,
    benjamini_hochberg,
    holm_correct,
    positivity_report,
)

cfg = load_config(f'{REPO}/config/study_config.yaml')
ROOT = Path(cfg.paths['drive_root'])
SCORES = ROOT / cfg.paths['scores']
ANALYSIS = ROOT / cfg.paths['analysis']
ANALYSIS.mkdir(parents=True, exist_ok=True)

PRIMARY_ARM = cfg.analysis['calibration'].get('primary_arm', 'global')
REFERENCE = cfg.analysis['standardization']['reference_mix']
BOOTSTRAP = cfg.analysis['bootstrap']['replicates']
CI = cfg.analysis['bootstrap']['ci']
MARGIN = cfg.analysis['outcomes']['independence_margin']
POSITIVITY = cfg.analysis['positivity']

tiers = pd.read_csv(ANALYSIS / f'label_tiers_{cfg.analysis_hash}.csv')
PRIMARY = tiers.loc[tiers.tier == 'primary', 'label'].tolist()
EXPLORATORY = tiers.loc[tiers.tier == 'exploratory', 'label'].tolist()

print('calibration arm :', PRIMARY_ARM, '(view-conditional is secondary)')
print('reference mix :', REFERENCE)
print('bootstrap :', f'{BOOTSTRAP:,} patient replicates, {CI:.0%} intervals')
print('independence margin:', MARGIN, '- the whole interval must lie inside it')
print('positivity :', POSITIVITY)
print('primary family :', PRIMARY)
print('exploratory :', EXPLORATORY)

In [ ]:
# --- load decisions -----------------------------------------------------------
decisions = {}
for site in cfg.training_sites:
    path = SCORES / f'decisions_{site}_{PRIMARY_ARM}_{cfg.model_hash}.parquet'
    decisions[site] = pd.read_parquet(path)
    frame = decisions[site]
    print(f'{site:<10} {len(frame):>7,} patients   '
          f'AP {(frame.view == "AP").mean():.1%}   '
          f'female {(frame.sex == "Female").mean():.1%}')

# the binary age contrast, from the fixed clinical threshold
for site, frame in decisions.items():
    frame['age_group'] = np.where(frame['age'] < cfg.primary_age_threshold,
                                  f'<{cfg.primary_age_threshold}',
                                  f'>={cfg.primary_age_threshold}')

CONTRASTS = {
    'sex': ('Female', 'Male'),
    'age_group': (f'<{cfg.primary_age_threshold}', f'>={cfg.primary_age_threshold}'),
}
print('\ncontrasts:', CONTRASTS)

In [ ]:
# --- compute overall FNR and flagged rate --------------------------------------
from pxr.stats.fairness import false_negative_rate

rows = []
for site, frame in decisions.items():
    for label in PRIMARY + EXPLORATORY:
        positives = int((frame[f'{label}_true'] == 1).sum())
        rows.append({
            'site': site, 'label': label, 'positives': positives,
            'FNR': round(false_negative_rate(frame, label), 4),
            'flagged': round(float((frame[f'{label}_predicted'] == 1).mean()), 4),
        })
overall = pd.DataFrame(rows)
print(overall.to_string(index=False))
print(f'\nThreshold was set for {1 - cfg.analysis["threshold_rule"]["fixed_sensitivity_target"]:.0%} '
      'FNR on validation; test FNR should sit near that.')

In [ ]:
# --- compute stratified analyses -----------------------------------------------
results = {}
for site, frame in decisions.items():
    for label in PRIMARY + EXPLORATORY:
        for stratum, levels in CONTRASTS.items():
            other = [s for s in CONTRASTS if s != stratum]
            results[(site, label, stratum)] = analyse_label(
                frame, label, stratum, levels,
                site=site, adjust_for=other,
                reference=REFERENCE, replicates=BOOTSTRAP, ci=CI,
                seed=cfg.splits['seed'],
            )
    print(f'{site}: {len([k for k in results if k[0] == site])} analyses complete')

In [ ]:
# --- collate results -----------------------------------------------------------
rows = []
for (site, label, stratum), result in results.items():
    if label not in PRIMARY:
        continue
    rows.append({'site': site, 'label': label, 'contrast': stratum, 'view': 'both',
                 'gap': result.raw.gap, 'ci_low': result.raw.ci_low,
                 'ci_high': result.raw.ci_high, 'n_a': result.raw.n_a,
                 'n_b': result.raw.n_b, 'significant': result.raw.significant})
    for estimate in result.within_view:
        rows.append({'site': site, 'label': label, 'contrast': stratum,
                     'view': estimate.view, 'gap': estimate.gap,
                     'ci_low': estimate.ci_low, 'ci_high': estimate.ci_high,
                     'n_a': estimate.n_a, 'n_b': estimate.n_b,
                     'significant': estimate.significant})

gaps = pd.DataFrame(rows)
for site in cfg.training_sites:
    print(f'=== {site} ===')
    print(gaps[gaps.site == site].drop(columns='site').round(4).to_string(index=False))
    print()

In [ ]:
# --- collate standardised results ----------------------------------------------
rows = [r.standardised.as_row() for (s, l, st), r in results.items()
        if l in PRIMARY and r.standardised is not None]
standardised = pd.DataFrame(rows)

for site in cfg.training_sites:
    block = standardised[standardised.site == site]
    print(f'=== {site} ===')
    print(block[['label', 'stratum', 'gap_raw', 'gap_standardised', 'delta',
                 'delta_ci_low', 'delta_ci_high', 'verdict',
                 'positivity_ok']].round(4).to_string(index=False))
    print()

In [ ]:
from dataclasses import replace

grid = cfg.analysis['outcomes']['independence_margin_sensitivity']
rows = []
for (site, label, stratum), result in results.items():
    if label not in PRIMARY or result.standardised is None:
        continue
    verdicts = {m: replace(result.standardised, equivalence_margin=m).verdict
                for m in grid}
    rows.append({'site': site, 'label': label, 'contrast': stratum,
                 **{f'margin={m}': v for m, v in verdicts.items()},
                 'stable': len(set(verdicts.values())) == 1})
sensitivity = pd.DataFrame(rows)
print(sensitivity.to_string(index=False))
print(f'\nverdict stable across the grid in '
      f'{int(sensitivity.stable.sum())} of {len(sensitivity)} analyses')

In [ ]:
# --- compute positivity reports ------------------------------------------------
for site, frame in decisions.items():
    for label in PRIMARY:
        for stratum, levels in CONTRASTS.items():
            ok, note, detail = positivity_report(
                frame, label, stratum, levels, reference=REFERENCE,
                min_positives=POSITIVITY['min_positives_per_cell'],
                max_reweight_distance=POSITIVITY['max_reweight_distance'],
            )
            flag = '' if ok else '   <-- FLAGGED'
            print(f'{site:<10} {label:<18} {stratum:<10}{flag}')
            print(detail.to_string(index=False))
            if not ok:
                print(f'{note}')
            print()

In [ ]:
# --- visualise standardised results --------------------------------------------
import matplotlib.pyplot as plt

primary_rows = standardised[standardised.label.isin(PRIMARY)]
fig, axes = plt.subplots(1, len(cfg.training_sites), figsize=(11, 4), sharey=True)
for ax, site in zip(np.atleast_1d(axes), cfg.training_sites):
    block = primary_rows[primary_rows.site == site]
    y = np.arange(len(block))
    ax.errorbar(block.delta, y,
                xerr=[block.delta - block.delta_ci_low,
                      block.delta_ci_high - block.delta],
                fmt='o', capsize=3)
    ax.axvline(0, ls='--', c='grey', lw=1)
    ax.set_yticks(y)
    ax.set_yticklabels([f'{r.label}\n{r.stratum}' for r in block.itertuples()], fontsize=8)
    ax.set_xlabel('ΔGap  (raw − standardised)')
    ax.set_title(site)
plt.suptitle('Right of zero: acquisition accounts for part of the disparity')
plt.tight_layout(); plt.show()

In [ ]:
# --- collate interaction models ------------------------------------------------
frames = [r.interaction for r in results.values()
          if r.interaction is not None and r.label in PRIMARY]
interactions = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

if len(interactions):
    terms = interactions[interactions.is_interaction]
    print(terms[['site', 'label', 'term', 'odds_ratio',
                 'ci_low', 'ci_high', 'p_value']].round(4).to_string(index=False))
else:
    print('no interaction models were fitted')

notes = [(k, n) for k, r in results.items() for n in r.notes]
if notes:
    print('\nnotes:')
    for key, note in notes:
        print(f'{key}: {note}')

In [ ]:
# --- collate interaction models ------------------------------------------------
if len(interactions):
    primary_terms = interactions[interactions.is_interaction
                                 & interactions.label.isin(PRIMARY)].copy()
    for site in cfg.training_sites:
        block = primary_terms[primary_terms.site == site]
        if block.empty:
            continue
        corrected = holm_correct(block.p_value.tolist(), alpha=1 - CI)
        out = block[['label', 'term', 'p_value']].reset_index(drop=True)
        out['p_holm'] = corrected.p_adjusted
        out['reject'] = corrected.reject
        print(f'=== {site}: primary family (Holm) ===')
        print(out.round(4).to_string(index=False))
        print()

    exploratory_terms = interactions[interactions.is_interaction
                                     & interactions.label.isin(EXPLORATORY)]
    if len(exploratory_terms):
        corrected = benjamini_hochberg(exploratory_terms.p_value.tolist(), alpha=1 - CI)
        out = exploratory_terms[['site', 'label', 'p_value']].reset_index(drop=True)
        out['p_bh'] = corrected.p_adjusted
        out['reject'] = corrected.reject
        print('=== exploratory family (Benjamini-Hochberg) ===')
        print(out.round(4).to_string(index=False))

In [ ]:
# --- compute stratified analyses -----------------------------------------------
pivot = standardised[standardised.label.isin(PRIMARY)].pivot_table(
    index=['label', 'stratum'], columns='site',
    values=['delta', 'verdict'], aggfunc='first')
print(pivot.to_string())
print()

agree = []
for (label, stratum), block in standardised[standardised.label.isin(PRIMARY)].groupby(
        ['label', 'stratum']):
    verdicts = set(block.verdict)
    signs = {np.sign(d) for d in block.delta}
    agree.append({'label': label, 'contrast': stratum,
                  'same_verdict': len(verdicts) == 1,
                  'same_direction': len(signs) == 1,
                  'verdicts': ', '.join(sorted(verdicts))})
print(pd.DataFrame(agree).to_string(index=False))

In [ ]:
# --- save results --------------------------------------------------------------
gaps.to_csv(ANALYSIS / f'gaps_{cfg.analysis_hash}.csv', index=False)
standardised.to_csv(ANALYSIS / f'standardised_{cfg.analysis_hash}.csv', index=False)
if len(interactions):
    interactions.to_csv(ANALYSIS / f'interactions_{cfg.analysis_hash}.csv', index=False)
print('saved to', ANALYSIS)

In [ ]:
# --- integrity cell ------------------
print(f'analysis_hash : {cfg.analysis_hash}')
print(f'model_hash : {cfg.model_hash}')
print(f'calibration arm : {PRIMARY_ARM}')
print(f'reference mix : {REFERENCE}')
print(f'bootstrap : {BOOTSTRAP:,} patient replicates')
print(f'primary family : {PRIMARY}')
print()
for site in cfg.training_sites:
    block = standardised[(standardised.site == site) & standardised.label.isin(PRIMARY)]
    for row in block.itertuples():
        flag = '' if row.positivity_ok else '[positivity]'
        print(f'{site:<10} {row.label:<18} {row.stratum:<10} '
              f'ΔGap {row.delta:+.4f} [{row.delta_ci_low:+.4f}, {row.delta_ci_high:+.4f}] '
              f'{row.verdict}{flag}')